In [1]:
from google.colab import files

uploaded = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset.csv


In [2]:
import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")

print(df.head())
print("Shape:", df.shape)
print("Columns:", df.columns)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Shape: (50000, 2)
Columns: Index(['review', 'sentiment'], dtype='object')


In [3]:
df = df.sample(
    n=3000,
    random_state=42
).reset_index(drop=True)

print("Dataset size:", len(df))
print(df["sentiment"].value_counts())

Dataset size: 3000
sentiment
positive    1509
negative    1491
Name: count, dtype: int64


In [4]:
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print(df[["sentiment", "label"]].head())

  sentiment  label
0  positive      1
1  positive      1
2  negative      0
3  positive      1
4  negative      0


In [5]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["review"].values,
    df["label"].values,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training data:", len(X_train))
print("Testing data:", len(X_test))

Training data: 2400
Testing data: 600


In [6]:
!pip install tiktoken


In [7]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

print("Tokenizer:", tokenizer.name)

Tokenizer: cl100k_base


In [8]:
def tokenize_text(text):
    return tokenizer.encode(text)

In [9]:
sample_tokens = tokenize_text(X_train[0])

print("Number of tokens:", len(sample_tokens))
print("First 20 token IDs:", sample_tokens[:20])

Number of tokens: 53
First 20 token IDs: [11649, 1131, 40, 1093, 420, 5818, 1176, 315, 682, 1606, 433, 596, 1633, 1664, 3463, 315, 1131, 323, 1664, 497]


In [10]:
MAX_LENGTH = 200

def pad_or_truncate(tokens, max_length=MAX_LENGTH):

    if len(tokens) > max_length:
        return tokens[:max_length]

    else:
        return tokens + [0] * (max_length - len(tokens))

In [11]:
X_train_padded = [
    pad_or_truncate(tokenize_text(text))
    for text in X_train
]

X_test_padded = [
    pad_or_truncate(tokenize_text(text))
    for text in X_test
]

print("Training shape:", len(X_train_padded), len(X_train_padded[0]))
print("Testing shape:", len(X_test_padded), len(X_test_padded[0]))

Training shape: 2400 200
Testing shape: 600 200


In [12]:
import torch

X_train_tensor = torch.tensor(
    X_train_padded,
    dtype=torch.long
)

X_test_tensor = torch.tensor(
    X_test_padded,
    dtype=torch.long
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.long
)

print("X_train:", X_train_tensor.shape)
print("X_test:", X_test_tensor.shape)
print("y_train:", y_train_tensor.shape)
print("y_test:", y_test_tensor.shape)

X_train: torch.Size([2400, 200])
X_test: torch.Size([600, 200])
y_train: torch.Size([2400])
y_test: torch.Size([600])


In [13]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

Train batches: 38
Test batches: 10


In [14]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cuda


In [15]:
VOCAB_SIZE = max(
    max(tokens) for tokens in X_train_padded
) + 1

print("Vocabulary size:", VOCAB_SIZE)

Vocabulary size: 100253


In [16]:
import torch.nn as nn

class RNNClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=64
    ):
        super(RNNClassifier, self).__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            2
        )

    def forward(self, x):

        # Embedding
        embedded = self.embedding(x)

        # RNN
        output, hidden = self.rnn(embedded)

        # Last hidden state
        hidden = hidden[-1]

        # Linear output
        output = self.fc(hidden)

        return output

In [17]:
rnn_model = RNNClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=128,
    hidden_dim=64
).to(device)

print(rnn_model)

RNNClassifier(
  (embedding): Embedding(100253, 128)
  (rnn): RNN(128, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


In [18]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

rnn_optimizer = optim.Adam(
    rnn_model.parameters(),
    lr=0.001
)

print("Loss Function: CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning Rate: 0.001")

Loss Function: CrossEntropyLoss
Optimizer: Adam
Learning Rate: 0.001


In [19]:
EPOCHS = 5

rnn_train_losses = []

for epoch in range(EPOCHS):

    rnn_model.train()

    total_loss = 0

    for batch_x, batch_y in train_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # Clear gradients
        rnn_optimizer.zero_grad()

        # Forward pass
        predictions = rnn_model(batch_x)

        # Calculate loss
        loss = criterion(predictions, batch_y)

        # Backpropagation
        loss.backward()

        # Update weights
        rnn_optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    rnn_train_losses.append(average_loss)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}], "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/5], Loss: 0.7054
Epoch [2/5], Loss: 0.6715
Epoch [3/5], Loss: 0.6470
Epoch [4/5], Loss: 0.6176
Epoch [5/5], Loss: 0.5668


In [21]:
from sklearn.metrics import accuracy_score

In [22]:
rnn_model.eval()

rnn_predictions = []
rnn_actual = []

with torch.no_grad():

    for batch_x, batch_y in test_loader:

        batch_x = batch_x.to(device)

        outputs = rnn_model(batch_x)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        rnn_predictions.extend(
            predictions.cpu().numpy()
        )

        rnn_actual.extend(
            batch_y.numpy()
        )

rnn_accuracy = accuracy_score(
    rnn_actual,
    rnn_predictions
)

print(f"RNN Test Accuracy: {rnn_accuracy * 100:.2f}%")

RNN Test Accuracy: 47.83%


In [23]:
custom_reviews = [
    "This movie was absolutely fantastic and I really enjoyed it.",
    "The movie was boring and a complete waste of time.",
    "The acting was excellent and the story was amazing."
]

In [24]:
def predict_rnn(review):

    tokens = tokenize_text(review)

    tokens = pad_or_truncate(
        tokens,
        MAX_LENGTH
    )

    tensor = torch.tensor(
        [tokens],
        dtype=torch.long
    ).to(device)

    rnn_model.eval()

    with torch.no_grad():

        output = rnn_model(tensor)

        prediction = torch.argmax(
            output,
            dim=1
        ).item()

    return "Positive" if prediction == 1 else "Negative"

In [25]:
print("RNN Predictions:\n")

for review in custom_reviews:

    prediction = predict_rnn(review)

    print("Review:", review)
    print("RNN Prediction:", prediction)
    print()

RNN Predictions:

Review: This movie was absolutely fantastic and I really enjoyed it.
RNN Prediction: Negative

Review: The movie was boring and a complete waste of time.
RNN Prediction: Negative

Review: The acting was excellent and the story was amazing.
RNN Prediction: Negative



In [26]:
class LSTMClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        hidden_dim=64
    ):
        super(LSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            2
        )

    def forward(self, x):

        # Embedding
        embedded = self.embedding(x)

        # LSTM
        output, (hidden, cell) = self.lstm(embedded)

        # Last hidden state
        hidden = hidden[-1]

        # Linear output
        output = self.fc(hidden)

        return output

In [27]:
lstm_model = LSTMClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=128,
    hidden_dim=64
).to(device)

print(lstm_model)

LSTMClassifier(
  (embedding): Embedding(100253, 128)
  (lstm): LSTM(128, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


In [28]:
lstm_criterion = nn.CrossEntropyLoss()

lstm_optimizer = optim.Adam(
    lstm_model.parameters(),
    lr=0.001
)

print("Loss Function: CrossEntropyLoss")
print("Optimizer: Adam")
print("Learning Rate: 0.001")

Loss Function: CrossEntropyLoss
Optimizer: Adam
Learning Rate: 0.001


In [29]:
EPOCHS = 5

lstm_train_losses = []

for epoch in range(EPOCHS):

    lstm_model.train()

    total_loss = 0

    for batch_x, batch_y in train_loader:

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # Clear gradients
        lstm_optimizer.zero_grad()

        # Forward pass
        predictions = lstm_model(batch_x)

        # Calculate loss
        loss = lstm_criterion(
            predictions,
            batch_y
        )

        # Backpropagation
        loss.backward()

        # Update weights
        lstm_optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    lstm_train_losses.append(average_loss)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}], "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/5], Loss: 0.6965
Epoch [2/5], Loss: 0.6807
Epoch [3/5], Loss: 0.6642
Epoch [4/5], Loss: 0.6313
Epoch [5/5], Loss: 0.5655


In [30]:
lstm_model.eval()

lstm_predictions = []
lstm_actual = []

with torch.no_grad():

    for batch_x, batch_y in test_loader:

        batch_x = batch_x.to(device)

        outputs = lstm_model(batch_x)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        lstm_predictions.extend(
            predictions.cpu().numpy()
        )

        lstm_actual.extend(
            batch_y.numpy()
        )

lstm_accuracy = accuracy_score(
    lstm_actual,
    lstm_predictions
)

print(
    f"LSTM Test Accuracy: "
    f"{lstm_accuracy * 100:.2f}%"
)

LSTM Test Accuracy: 52.33%


In [31]:
def predict_lstm(review):

    tokens = tokenize_text(review)

    tokens = pad_or_truncate(
        tokens,
        MAX_LENGTH
    )

    tensor = torch.tensor(
        [tokens],
        dtype=torch.long
    ).to(device)

    lstm_model.eval()

    with torch.no_grad():

        output = lstm_model(tensor)

        prediction = torch.argmax(
            output,
            dim=1
        ).item()

    return "Positive" if prediction == 1 else "Negative"

In [32]:
print("LSTM Predictions:\n")

for review in custom_reviews:

    prediction = predict_lstm(review)

    print("Review:", review)
    print("LSTM Prediction:", prediction)
    print()

LSTM Predictions:

Review: This movie was absolutely fantastic and I really enjoyed it.
LSTM Prediction: Negative

Review: The movie was boring and a complete waste of time.
LSTM Prediction: Negative

Review: The acting was excellent and the story was amazing.
LSTM Prediction: Negative

